In [1]:
!curl -fsSL https://ollama.com/install.sh | sh


>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:

import time

!pkill -f ollama
!ollama serve > /dev/null 2>&1 &
time.sleep(5)

!ollama pull nomic-embed-text
!ollama pull qwen2


In [7]:
!ollama show --modelfile qwen2 > Modelfile
# PARAMETER num_ctx 32000


In [8]:
!ollama create -f Modelfile qwen2:ctx32k

In [9]:
import json
import xml.etree.ElementTree as ET
import requests
import time
from typing import Dict, List, Tuple, Optional
import subprocess

# !ollama serve
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

OLLAMA_API = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2:ctx32k"

def query_ollama(prompt: str, model: str = MODEL_NAME) -> str:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }

    try:
        response = requests.post(OLLAMA_API, json=payload)
        response.raise_for_status()
        return response.json()["response"].strip()
    except Exception as e:
        print(f"Error querying Ollama: {e}")
        return ""

def load_graphml(file_path: str) -> ET.ElementTree:
    tree = ET.parse(file_path)
    return tree

def load_kv_store(file_path: str) -> Dict:
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def get_full_doc_id_from_chunk(chunk_id: str, kv_store_chunks: Dict) -> Optional[str]:
    """
    Look up the chunk_id in kv_store_text_chunks.json to get the full_doc_id
    """
    if chunk_id in kv_store_chunks:
        return kv_store_chunks[chunk_id].get('full_doc_id')
    return None

def get_node_data(node: ET.Element, ns: str) -> Dict:
    node_id = node.get('id')
    data = {}
    for data_elem in node.findall(f'{ns}data'):
        key = data_elem.get('key')
        data[key] = data_elem.text

    return {
        'id': node_id,
        'entity_type': data.get('d0', ''),
        'description': data.get('d1', ''),
        'source_id': data.get('d2', '')
    }

def get_edge_data(edge: ET.Element, ns: str) -> Dict:
    source = edge.get('source')
    target = edge.get('target')
    data = {}
    for data_elem in edge.findall(f'{ns}data'):
        key = data_elem.get('key')
        data[key] = data_elem.text

    return {
        'source': source,
        'target': target,
        'weight': data.get('d4', ''),
        'description': data.get('d5', ''),
        'source_id': data.get('d6', ''),
        'order': data.get('d7', '')
    }

def classify_relationship_initial(source_node: Dict, target_node: Dict, edge_desc: str) -> Tuple[str, bool]:

    prompt = f"""You are analyzing relationships between educational concepts to build a tutoring knowledge graph.

Source Concept: {source_node['id'].strip('"')}
Type: {source_node['entity_type'].strip('"')}
Description: {source_node['description'].strip('"')}

Target Concept: {target_node['id'].strip('"')}
Type: {target_node['entity_type'].strip('"')}
Description: {target_node['description'].strip('"')}

Relationship Description: {edge_desc.strip('"')}

Classify this relationship as ONE of the following:
- PREREQUISITE_FOR: The source concept must be learned/understood before the target concept
- EXPLAINS: The source (typically a resource, method, or material) explains or teaches the target concept
- NEAR_TRANSFER: The source and target concepts share similar cognitive skills or problem-solving approaches
- NONE: None of the above categories apply

IMPORTANT: If you notice that an EXAMPLE is involved in this relationship (e.g., the source is an example/case study of the target, or vice versa), mention "EXAMPLE_INVOLVED" in your response.

Respond in EXACTLY this format:
RELATIONSHIP: [your classification]
EXAMPLE_INVOLVED: [YES or NO]
REASONING: [brief explanation]"""

    response = query_ollama(prompt)

    relationship = "NONE"
    has_example = False

    lines = response.split('\n')
    for line in lines:
        if line.startswith("RELATIONSHIP:"):
            rel = line.split(":", 1)[1].strip()
            if rel in ["PREREQUISITE_FOR", "EXPLAINS", "NEAR_TRANSFER", "NONE"]:
                relationship = rel
        elif line.startswith("EXAMPLE_INVOLVED:"):
            has_example = "YES" in line.upper()

    return relationship, has_example

def classify_relationship_with_text(source_node: Dict, target_node: Dict, edge_desc: str, text_chunk: str) -> str:

    prompt = f"""You are analyzing relationships between educational concepts to build a tutoring knowledge graph.

Source Concept: {source_node['id'].strip('"')}
Type: {source_node['entity_type'].strip('"')}
Description: {source_node['description'].strip('"')}

Target Concept: {target_node['id'].strip('"')}
Type: {target_node['entity_type'].strip('"')}
Description: {target_node['description'].strip('"')}

Relationship Description: {edge_desc.strip('"')}

Original Text Chunk:
{text_chunk[:2000]}

Classify this relationship as ONE of the following:
- PREREQUISITE_FOR: The source concept must be learned/understood before the target concept
- EXPLAINS: The source (typically a resource, method, or material) explains or teaches the target concept
- EXAMPLE_OF: The source is a concrete example, case study, or specific instance of the target concept
- NEAR_TRANSFER: The source and target concepts share similar cognitive skills or problem-solving approaches
- NONE: None of the above categories apply

Look for evidence in the text such as:
- "prerequisite", "before", "requires understanding", "builds on" → PREREQUISITE_FOR
- "explains", "tutorial", "course", "teaches" → EXPLAINS
- "example", "case study", "instance", "for example" → EXAMPLE_OF
- "similar to", "related approach", "analogous" → NEAR_TRANSFER

Respond in EXACTLY this format:
RELATIONSHIP: [your classification]
CONFIDENCE: [HIGH, MEDIUM, or LOW]
REASONING: [brief explanation with evidence from text]"""

    response = query_ollama(prompt)

    relationship = "NONE"

    lines = response.split('\n')
    for line in lines:
        if line.startswith("RELATIONSHIP:"):
            rel = line.split(":", 1)[1].strip()
            if rel in ["PREREQUISITE_FOR", "EXPLAINS", "EXAMPLE_OF", "NEAR_TRANSFER", "NONE"]:
                relationship = rel
                break

    return relationship

def create_output_graphml(nodes: List[ET.Element], edges_with_relationships: List[Tuple[ET.Element, str]],
                          output_path: str, ns: str):

    root = ET.Element('graphml')
    root.set('xmlns', 'http://graphml.graphdrawing.org/xmlns')
    root.set('xmlns:xsi', 'http://www.w3.org/2001/XMLSchema-instance')
    root.set('xsi:schemaLocation', 'http://graphml.graphdrawing.org/xmlns http://graphml.graphdrawing.org/xmlns/1.0/graphml.xsd')


    ET.SubElement(root, 'key', id='d0', attrib={'for': 'node', 'attr.name': 'entity_type', 'attr.type': 'string'})
    ET.SubElement(root, 'key', id='d1', attrib={'for': 'node', 'attr.name': 'description', 'attr.type': 'string'})
    ET.SubElement(root, 'key', id='d2', attrib={'for': 'node', 'attr.name': 'source_id', 'attr.type': 'string'})
    ET.SubElement(root, 'key', id='d3', attrib={'for': 'node', 'attr.name': 'clusters', 'attr.type': 'string'})

    ET.SubElement(root, 'key', id='d4', attrib={'for': 'edge', 'attr.name': 'weight', 'attr.type': 'double'})
    ET.SubElement(root, 'key', id='d5', attrib={'for': 'edge', 'attr.name': 'description', 'attr.type': 'string'})
    ET.SubElement(root, 'key', id='d6', attrib={'for': 'edge', 'attr.name': 'source_id', 'attr.type': 'string'})
    ET.SubElement(root, 'key', id='d7', attrib={'for': 'edge', 'attr.name': 'order', 'attr.type': 'long'})
    ET.SubElement(root, 'key', id='d8', attrib={'for': 'edge', 'attr.name': 'relationship_type', 'attr.type': 'string'})

    graph = ET.SubElement(root, 'graph', edgedefault='undirected')

    for node in nodes:
        graph.append(node)

    for edge, rel_type in edges_with_relationships:
        new_edge = ET.SubElement(graph, 'edge', source=edge.get('source'), target=edge.get('target'))

        for data_elem in edge.findall(f'{ns}data'):
            new_data = ET.SubElement(new_edge, 'data', key=data_elem.get('key'))
            new_data.text = data_elem.text

        rel_data = ET.SubElement(new_edge, 'data', key='d8')
        rel_data.text = rel_type

    tree = ET.ElementTree(root)
    ET.indent(tree, space='  ')
    tree.write(output_path, encoding='utf-8', xml_declaration=True)

def main():
    graphml_path = 'graph_chunk_entity_relation.graphml'
    kv_store_chunks_path = 'kv_store_text_chunks.json'
    kv_store_full_docs_path = 'kv_store_full_docs.json'
    output_path = 'pedagogical_knowledge_graph.graphml'

    tree = load_graphml(graphml_path)
    root = tree.getroot()

    ns = '{http://graphml.graphdrawing.org/xmlns}'

    kv_store_chunks = load_kv_store(kv_store_chunks_path)
    kv_store_full_docs = load_kv_store(kv_store_full_docs_path)

    graph = root.find(f'{ns}graph')
    nodes = list(graph.findall(f'{ns}node'))
    edges = list(graph.findall(f'{ns}edge'))

    node_lookup = {}
    for node in nodes:
        node_data = get_node_data(node, ns)
        node_lookup[node_data['id']] = node_data

    print(f"Found {len(nodes)} nodes and {len(edges)} edges")
    print(f"Processing {len(edges)} relationships...\n")

    edges_with_relationships = []

    for i, edge in enumerate(edges):
        edge_data = get_edge_data(edge, ns)
        source_node = node_lookup.get(edge_data['source'])
        target_node = node_lookup.get(edge_data['target'])

        if not source_node or not target_node:
            edges_with_relationships.append((edge, "NONE"))
            continue

        print(f"[{i+1}/{len(edges)}] Processing: {source_node['id'].strip('\"')} -> {target_node['id'].strip('\"')}")

        relationship, has_example = classify_relationship_initial(
            source_node, target_node, edge_data['description']
        )
        print(relationship)

        if has_example:
            chunk_id = edge_data.get('source_id', '')

            full_doc_id = get_full_doc_id_from_chunk(chunk_id, kv_store_chunks)

            if full_doc_id and full_doc_id in kv_store_full_docs:
                text_chunk = kv_store_full_docs[full_doc_id].get('content', '')
                relationship = classify_relationship_with_text(
                    source_node, target_node, edge_data['description'], text_chunk
                )
                print(f"  Final: {relationship}")
            else:
                if not full_doc_id:
                    print(f"  Warning: Could not find full_doc_id for chunk {chunk_id}")
                else:
                    print(f"  Warning: Could not find text for document {full_doc_id}")

        edges_with_relationships.append((edge, relationship))
        print()

        time.sleep(0.5)

    create_output_graphml(nodes, edges_with_relationships, output_path, ns)



In [ ]:
main()